In [ ]:
# =========================
# SafeSeniors - Small NN Pipeline
# Fall Detection using Accelerometer + Gyroscope
# =========================

import os
import json
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score
)

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

In [ ]:
# =========================
# 1. CONFIG
# =========================

DATA_PATH = "/Users/dhanujiamanda/Documents/IIT/Stage 3/Edge/CW/SafeSeniors/data/full_dataset.csv"
OUTPUT_DIR = "outputs"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Raw sensor columns expected in CSV
ACC_COLS = ["acc_x", "acc_y", "acc_z"]
GYRO_COLS = ["gyro_x", "gyro_y", "gyro_z"]
LABEL_COL = "label"

# Label mapping
# Adjust if your labels are different
LABEL_MAP = {
    "Normal": 0,
    "Fall": 1
}

RANDOM_STATE = 42
TEST_SIZE = 0.15
VAL_SIZE = 0.15
BATCH_SIZE = 32
EPOCHS = 50

In [ ]:
# =========================
# 2. LOAD DATA
# =========================

df = pd.read_csv(DATA_PATH)

required_cols = ACC_COLS + GYRO_COLS + [LABEL_COL]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

print("Data shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df.head())


# =========================
# 3. CLEAN + LABEL ENCODE
# =========================

df = df.dropna(subset=required_cols).copy()

# Convert label strings to numeric
if df[LABEL_COL].dtype == object:
    df[LABEL_COL] = df[LABEL_COL].map(LABEL_MAP)

if df[LABEL_COL].isna().any():
    bad_labels = df[df[LABEL_COL].isna()]
    raise ValueError(
        f"Found unmapped labels in '{LABEL_COL}'. "
        f"Check LABEL_MAP. Sample:\n{bad_labels.head()}"
    )

df[LABEL_COL] = df[LABEL_COL].astype(int)

print("\nLabel distribution:")
print(df[LABEL_COL].value_counts())


# =========================
# 4. FEATURE ENGINEERING
# =========================

def add_engineered_features(data: pd.DataFrame) -> pd.DataFrame:
    out = data.copy()

    # Acceleration magnitude
    out["acc_mag"] = np.sqrt(
        out["acc_x"]**2 + out["acc_y"]**2 + out["acc_z"]**2
    )

    # Gyroscope magnitude
    out["gyro_mag"] = np.sqrt(
        out["gyro_x"]**2 + out["gyro_y"]**2 + out["gyro_z"]**2
    )

    # Simple orientation proxies
    # These are rough engineered signals, useful for a small NN
    out["acc_xy_ratio"] = out["acc_x"] / (np.abs(out["acc_y"]) + 1e-6)
    out["acc_xz_ratio"] = out["acc_x"] / (np.abs(out["acc_z"]) + 1e-6)
    out["gyro_xy_ratio"] = out["gyro_x"] / (np.abs(out["gyro_y"]) + 1e-6)

    # Absolute values can help capture intensity
    out["abs_acc_x"] = np.abs(out["acc_x"])
    out["abs_acc_y"] = np.abs(out["acc_y"])
    out["abs_acc_z"] = np.abs(out["acc_z"])
    out["abs_gyro_x"] = np.abs(out["gyro_x"])
    out["abs_gyro_y"] = np.abs(out["gyro_y"])
    out["abs_gyro_z"] = np.abs(out["gyro_z"])

    return out

df = add_engineered_features(df)

feature_cols = [c for c in df.columns if c != LABEL_COL]
X = df[feature_cols].astype(np.float32)
y = df[LABEL_COL].astype(np.int32)

print("\nFeature columns:")
print(feature_cols)

In [ ]:
# =========================
# 5. TRAIN / VAL / TEST SPLIT
# =========================

# First split off test
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

# Then split train/val
val_relative_size = VAL_SIZE / (1.0 - TEST_SIZE)

X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=val_relative_size,
    random_state=RANDOM_STATE,
    stratify=y_trainval
)

print("\nSplit shapes:")
print("Train:", X_train.shape, y_train.shape)
print("Val:  ", X_val.shape, y_val.shape)
print("Test: ", X_test.shape, y_test.shape)


# =========================
# 6. SCALE FEATURES
# =========================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

joblib.dump(scaler, os.path.join(OUTPUT_DIR, "scaler.pkl"))
joblib.dump(feature_cols, os.path.join(OUTPUT_DIR, "feature_cols.pkl"))

with open(os.path.join(OUTPUT_DIR, "label_map.json"), "w") as f:
    json.dump(LABEL_MAP, f, indent=2)


# =========================
# 7. BUILD SMALL NN
# =========================

input_dim = X_train_scaled.shape[1]

model = Sequential([
    Input(shape=(input_dim,)),
    Dense(32, activation="relu"),
    Dropout(0.2),
    Dense(16, activation="relu"),
    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()


# =========================
# 8. TRAIN
# =========================

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),
    ModelCheckpoint(
        filepath=os.path.join(OUTPUT_DIR, "best_model.keras"),
        monitor="val_loss",
        save_best_only=True
    )
]

history = model.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)


# =========================
# 9. EVALUATE
# =========================

y_prob = model.predict(X_test_scaled).ravel()
y_pred = (y_prob >= 0.5).astype(int)

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)

print("\n=== TEST METRICS ===")
print(f"Accuracy : {acc:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=4))


# =========================
# 10. SAVE MODEL
# =========================

saved_model_dir = os.path.join(OUTPUT_DIR, "saved_model")
model.export(saved_model_dir)  # TF 2.13+ style


# =========================
# 11. CONVERT TO TFLITE
# =========================

converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

tflite_path = os.path.join(OUTPUT_DIR, "fall_detection_model.tflite")
with open(tflite_path, "wb") as f:
    f.write(tflite_model)

print(f"\nTFLite model saved to: {tflite_path}")


# =========================
# 12. QUICK TFLITE TEST
# =========================

interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

sample = X_test_scaled[:1].astype(np.float32)

interpreter.set_tensor(input_details[0]["index"], sample)
interpreter.invoke()
tflite_output = interpreter.get_tensor(output_details[0]["index"])

print("\nSample TFLite prediction:", tflite_output)


# =========================
# 13. SIMPLE INFERENCE FUNCTION
# =========================

def predict_single_sample(raw_sample: dict, scaler_obj, keras_model, feature_names: list) -> dict:
    """
    raw_sample example:
    {
        "acc_x": 1.2,
        "acc_y": 0.3,
        "acc_z": 9.5,
        "gyro_x": 0.1,
        "gyro_y": 0.2,
        "gyro_z": 0.3
    }
    """
    sample_df = pd.DataFrame([raw_sample])
    sample_df = add_engineered_features(sample_df)

    sample_X = sample_df[feature_names].astype(np.float32)
    sample_X_scaled = scaler_obj.transform(sample_X)

    prob = float(keras_model.predict(sample_X_scaled, verbose=0).ravel()[0])
    pred = int(prob >= 0.5)
    label_name = "Fall" if pred == 1 else "Normal"

    return {
        "prediction": pred,
        "label": label_name,
        "probability_fall": prob
    }


# Example test inference
example_input = {
    "acc_x": 2.1,
    "acc_y": 1.8,
    "acc_z": 8.5,
    "gyro_x": 0.6,
    "gyro_y": 0.4,
    "gyro_z": 0.7
}

result = predict_single_sample(example_input, scaler, model, feature_cols)
print("\nExample inference:")
print(result)